In [ ]:
from numpyro_model import *
from model_new import *

import numpy as np
import scipy as sp
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro
import jax.scipy as jsp
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
jax.config.update("jax_enable_x64", False)
jax.config.update('jax_log_compiles', False)

In [ ]:

Vc0 = 240.
Nknots = 10

path = '/content/drive/MyDrive/'
df_mockdata = pd.read_csv(path+'radial_migration_kernel/mock_sample/L_conditioned/mock_data3.csv')#_errorfree_L@10
feh_range = [-1.5,0.3]
logage_range = np.log10([0.3,12])
Rg_range = [9,11]
read_file = path+f'radial_migration_kernel/mock_sample/L_conditioned/minimization_result_withMHgrad_{Nknots}knots2.npy'
save_file = path+f'radial_migration_kernel/mock_sample/L_conditioned/results_withMHgrad_{Nknots}knots_OnMockdata3_2.pkl'#L@10_3
save_sampler = path+f"radial_migration_kernel/mock_sample/L_conditioned/results_withMHgrad_{Nknots}knots_idata_OnMockdata3.nc"
#

# df_mockdata = pd.read_csv(path+'radial_migration_kernel/mock_sample/with_bar/mock_data1.csv')#_errorfree_L@10
# feh_range = [-1.5,0.3]
# logage_range = np.log10([0.3,12])
# Rg_range = [6,7]
# read_file = path+f'radial_migration_kernel/mock_sample/with_bar/minimization_result_withMHgrad_{Nknots}knots1.npy'
# save_file = path+f'radial_migration_kernel/mock_sample/with_bar/results_withMHgrad_{Nknots}knots_OnMockdata1_1.pkl'#L@10_3
# save_sampler = path+f"radial_migration_kernel/mock_sample/with_bar/results_withMHgrad_{Nknots}knots_idata_OnMockdata1.nc"

df_mockdata = df_mockdata.sample(frac = 0.3, random_state = 42)


F, logage, L = df_mockdata['MH'], df_mockdata['log_age'], df_mockdata['Lz']
sigma_F, sigma_logage, sigma_L = df_mockdata['sigma_MH'], df_mockdata['sigma_logage'], df_mockdata['sigma_Lz']

e_feh_median = np.amax([np.median(sigma_F), 0.02])
e_log10age_median = np.amax([np.median(sigma_logage), 0.02])
e_Lz_median = np.amax([np.median(sigma_L), 20])
print('Median errors: e_feh_median = ', e_feh_median,
      'e_log10age_median = ', e_log10age_median,
      'e_Lz_median = ', e_Lz_median)


# data_grid = binning_with_different_sigma(df_mockdata,
#                                          MH_sigma_level=[0,0.02,0.05,0.1],
#                                          logage_sigma_level=[0,0.02,0.05,0.1],
#                                          feh_range=feh_range,
#                                          logage_range=logage_range,
#                                          Rg_range = Rg_range,
#                                          Vc0 = 240.,)

#=====================================================================================

data_grid = {
    'MH': jnp.array(F),
    'log_age': jnp.array(logage),
    'Lz': jnp.array(L),
    'sigma_MH': jnp.array(sigma_F),
    'sigma_logage': jnp.array(sigma_logage),
    'sigma_Lz': jnp.array(sigma_L),
    'Nstars': jnp.ones_like(F),  # Assuming equal weights for simplicity
}
#=====================================================================================

print('Number of grid points:', len(data_grid['MH']))
print('Number of total stars:', jnp.sum(data_grid['Nstars']))

N_sample = int(1e3)
# F_centre_for_sampling, F_scale_for_sampling = -0.5,  0.5
R_scale_for_sampling = 4
R_scale_at_0, R_scale_at_12 = 4, 1
F_centre_at_0, F_centre_at_12 = -0.1, -0.7
F_scale_at_0, F_scale_at_12 = 0.1, 0.7

data_generated = generate_sample_for_MC_integration_withprob_samenumdenom(data_grid,
                                    R_scale_at_0 = R_scale_at_0, R_scale_at_12 = R_scale_at_12,
                                    F_centre_at_0=F_centre_at_0, F_centre_at_12=F_centre_at_12,
                                    F_scale_at_0=F_scale_at_0, F_scale_at_12=F_scale_at_12,
                                    N_sample = N_sample)

# data_generated = generate_sample_for_MC_integration_withprob(data_grid, R_scale_for_sampling = R_scale_for_sampling,
#                                                     F_centre_at_0=F_centre_at_0, F_centre_at_12=F_centre_at_12,
#                                                     F_scale_at_0=F_scale_at_0, F_scale_at_12=F_scale_at_12,
#                                                     N_sample = N_sample)

# data_generated['weights'] = jnp.ones(len(F))
data_generated['weights'] = data_grid['Nstars']
shape = data_generated['age_sample'].shape
age = data_generated['age_sample'].reshape(-1)
data_generated['MH_max_sample'] = MH_evolution_Lu24(age, 0).reshape(shape)

# aux_params = {'R_scale_for_sampling':R_scale_for_sampling,
#               'F_scale_for_sampling':F_scale_for_sampling,
#               'F_centre_for_sampling':F_centre_for_sampling}  # Auxiliary parameters for the model
aux_params = {'ln_MH_grad_0': -2.66, 'tol': 5e-2, 'Vc0': 240.}  # Auxiliary parameters for the model
aux_params['aux_knots'] = generate_aux_knots(Nknots=Nknots, age_max=12.)#jnp.linspace(0.,15.,Nknots)
print('aux knots:', aux_params['aux_knots'])
# aux_params = {}


def lnG(x, mu, s):
    """
    Log of Gaussian distribution

    Args:
        x (float or array): Value(s) at which to evaluate the log of the Gaussian
        mu (float): Mean of the Gaussian
        s (float): Standard deviation of the Gaussian
    """
    return -.5*(x-mu)**2/s**2 - .5*jnp.log(2.*jnp.pi*s**2)

prior_scale_uniform = {'ln_Rdisk':[-2,2.5],
                       'ln_sigmaLz':[3., 6.3],
                       'MH_at_8':[-1.5, 0.2],
                        'ln_MH_grad':[-3.5, -1.5],
                       }
prior_scale_normal = {'ln_Rdisk':[0.5,0.5],
                      'ln_sigmaLz':[4.7, 0.5],
                      'MH_at_8':[-0.5, 0.5],
                      'ln_MH_grad':[-2.6, 0.5],
                      }

MH_max_param_normal_mean = [2., -0.7, -3.5, -2.5]
MH_max_param_normal_std = [0.5, 0.5, 1, 1]
MH_max_param_uniform_min = [0., -2, -5, -5]
MH_max_param_uniform_max = [2.7, 1.5, -0.6, -0.6]

@jax.jit
def log_prior(params):
    '''
    Log prior for the parameters
    '''
    lnPrior1 = lnRdisk_prior_normal(params)
    lnPrior2 = lnSigmaLz_prior_normal(params)
    lnPrior4 = ln_MH_grad_prior_normal(params)
    # lnPrior5 = ln_MH_grad_prior_uniform(params_S)
    lnPrior_smooth = smoothing_prior_withMHmodel2(params)

    return lnPrior1 + lnPrior2 + lnPrior4 + lnPrior_smooth

parameters = {
    'ln_Rdisk': numpyro.distributions.Normal(
        loc=prior_scale_normal['ln_Rdisk'][0]*jnp.ones(Nknots),
        scale=prior_scale_normal['ln_Rdisk'][1]*jnp.ones(Nknots),).expand([Nknots]),
    'ln_sigmaLz': numpyro.distributions.Normal(
        loc=prior_scale_normal['ln_sigmaLz'][0]*jnp.ones(Nknots),
        scale=prior_scale_normal['ln_sigmaLz'][1]*jnp.ones(Nknots),).expand([Nknots]),
    'ln_MH_grad': numpyro.distributions.Normal(
        loc=prior_scale_normal['ln_MH_grad'][0]*jnp.ones(Nknots),
        scale=prior_scale_normal['ln_MH_grad'][1]*jnp.ones(Nknots),).expand([Nknots]),
}


def logL_zero(params, data):
    # Return the log likelihood
    return 0


init_from_minimiser = True

n_warmup = 100
n_samples = 200
num_chains = 4
max_tree_depth = 7
target_accept_prob = 0.8
step_size = 1e-1
adapt_step_size = True
extra_fields = ('num_steps', 'adapt_state.step_size')
jit_model_args = True
# init_strategy=numpyro.infer.init_to_sample()

if init_from_minimiser:

    # file_name = path+f'radial_migration_kernel/mock_sample/L_conditioned/minimization_result_withMHgrad_{Nknots}knots4_binned.npy'#_L@10
    file_name = read_file
    minimiser_results = np.load(file_name)
    init_guess = {
        'ln_Rdisk': jnp.array(minimiser_results[:Nknots]),
        'ln_sigmaLz': jnp.array(minimiser_results[Nknots:2*Nknots]),
        'ln_MH_grad': jnp.array(minimiser_results[2*Nknots:3*Nknots]),
    }
    print('Initial guess from minimiser:', init_guess)
    init_strategy=numpyro.infer.initialization.init_to_value(values=init_guess)
else:
    init_strategy=numpyro.infer.init_to_sample()



print('model initialising...')

model = numpyro_model(logL_numpyro_withMHmodel3, parameters, data_generated, aux_parameters=aux_params,
                      expand_args=True, log_prior_fn=log_prior)#logL_numpyro, logL_zero

print('model initialised, and start running MCMC...')
model.run_mcmc(num_warmup=n_warmup, num_samples=n_samples, num_chains=num_chains,
                   init_strategy=init_strategy, max_tree_depth=max_tree_depth, step_size=step_size,
                   target_accept_prob=target_accept_prob, adapt_step_size=adapt_step_size,
                   chain_method="vectorized", extra_fields=extra_fields, jit_model_args=jit_model_args) # sequential, vectorized

print('MCMC finished, collecting samples...')
samples = model.samples()
print(samples)
# file = path+f'radial_migration_kernel/mock_sample/L_conditioned/Prior_distribution_{Nknots}knots.pkl'
file = save_file
with open(file,'wb') as f:
    pickle.dump(samples, f)

ef = model.mcmc.get_extra_fields(group_by_chain=True,
                                )
num_steps = ef['num_steps']    # shape (n_chains, n_warmup+n_samples)
print("Avg leapfrog steps per sample:",
      num_steps.mean())
print("Total lnL/grad calls ≃", num_steps.sum())

model.mcmc.print_summary()
